# Read Email Content

In [0]:
import os

volume_path = "/Volumes/projects/kindle/email_highlight/"
entries = sorted(
    [e for e in os.scandir(volume_path) if e.is_file()],
    key=lambda e: e.stat().st_mtime,
    reverse=True
)
print(entries)
file_path = entries[0].path
s3_file_name = entries[0].name
print(f"Triggered by file: {file_path}")
print(f"File name: {s3_file_name}")

with open(file_path, "r") as f:
    raw_email_content = f.read()

In [0]:
# # Manual override
# path = "/Volumes/projects/kindle/email_highlight/ijna6eno8aq52gf9hg7a04oe73cng15nnbkn6j81"
# s3_file_name = path.split('/')[-1]
# print(s3_file_name)

# with open(path, "r") as f:
#     raw_email_content = f.read()

# Extract Data from Email

In [0]:
import quopri
from bs4 import BeautifulSoup
import pandas as pd

# Decode quoted-printable encoding
decoded = quopri.decodestring(raw_email_content.encode()).decode("utf-8", errors="ignore")

soup = BeautifulSoup(decoded, "html.parser")

rows = []
current_book = None
current_section = None
pending_location = None

for tag in soup.find_all("div", class_=["bookTitle", "sectionHeading", "noteHeading", "noteText"]):
    cls = tag["class"][0]

    if cls == "bookTitle":
        current_book = tag.get_text(strip=True)
        current_section = None

    elif cls == "sectionHeading":
        current_section = tag.get_text(strip=True)

    elif cls == "noteHeading":
        pending_location = tag.get_text(strip=True)

    elif cls == "noteText":
        rows.append({
            "book_title": current_book,
            "section": current_section,
            "location_raw": pending_location,
            "highlight": tag.get_text(strip=True),
        })

df = pd.DataFrame(rows, columns=["book_title", "section", "location_raw", "highlight"])

# Add page_location and location columns after location_raw
df['page_location'] = df['location_raw'].apply(lambda x: x.split('-')[1].strip() if isinstance(x, str) and '-' in x else None)
df['location'] = df['location_raw'].apply(lambda x: x.split('Location')[1].strip() if isinstance(x, str) and 'Location' in x else None)

cols = ["book_title", "section", "location_raw", "page_location", "location", "highlight"]
df = df[cols]

df['s3_file_name'] = s3_file_name
display(df)

# Extract Book Title and Author from book_title

In [0]:
# Retrieve unique titles 
unique_titles = df['book_title'].unique().tolist()

print(unique_titles)

In [0]:
import json

def clean_titles_with_ai(messy_titles):
    # We create one big prompt to clean ALL unique titles at once 
    # Sending one block of text only results in one API request
    titles_string = "\n".join(messy_titles)
    
    prompt = f"""
    I have a list of messy book titles. For each one, extract the core 'title' and the primary 'author'.
    Remove: \ufeff, (Z-Library), translator names, and redundant repetitions.
    Return ONLY a JSON list of objects.
    
    Titles:
    {titles_string}
    """
    
    # ai_query() is a built-in Databricks function that bypasses the external API gateway
    # "databricks-gpt-5-2",
    query = f"""
    SELECT ai_query(
        "databricks-gpt-oss-120b",
        "{prompt}"
    )
    """
    
    # Run the query and parse the result
    raw_response = spark.sql(query).collect()[0][0]
    return json.loads(raw_response)

# Clean titles in one batch
cleaned_list = clean_titles_with_ai(unique_titles)
print(cleaned_list)

In [0]:
# # Create a mapping and update your dataframe
mapping = {m: c for m, c in zip(unique_titles, cleaned_list)}
temp_series = df['book_title'].map(mapping)

# Extract the book_title and author into their own columns
df['title'] = temp_series.apply(lambda x: x.get('title') if isinstance(x, dict) else None)
df['author'] = temp_series.apply(lambda x: x.get('author') if isinstance(x, dict) else None)

df_title_cleaned = df
display(df_title_cleaned)

# Extract Book Cover from Google API

In [0]:
import requests

def get_book_details(title, author):
    params = {
          "q": f'intitle:"{title}" inauthor:"{author}"',
          "maxResults": 1
      }
    url = "https://www.googleapis.com/books/v1/volumes"
    
    # Initialize a default "empty" result
    fallback_result = {
        "cover_url": None, 
        "published_date": None, 
        "genre": "Unknown", 
        "identifier": None
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if "items" in data:
            book_info = data["items"][0].get("volumeInfo", {})

            # 1. Safe Image Extraction with the Zoom Hack
            images = book_info.get('imageLinks', {})
            raw_url = images.get('thumbnail') or images.get('smallThumbnail')
            cover_url = None
            if raw_url:
                # Regex-free way to swap zoom levels
                cover_url = raw_url.replace("zoom=1", "zoom=3").replace("zoom=5", "zoom=3")
                cover_url = cover_url.replace("http://", "https://").replace("&edge=curl", "")

            # 2. Extract other fields safely using .get()
            return {
                "cover_url": cover_url,
                "published_date": book_info.get('publishedDate'),
                "genre": book_info.get("categories", ["Unknown"])[0],
                "identifier": book_info.get("industryIdentifiers")
            }
        
        # If "items" isn't in data, return the empty dictionary
        return fallback_result
        
    except Exception as e:
        print(f"Error fetching {title}: {e}")
        # Even if the internet cuts out or the API breaks, return the dictionary
        return fallback_result

# We use a unique list first to be polite to the API
unique_books = df_title_cleaned[['title', 'author']].drop_duplicates()

# Apply the function
details = unique_books.apply(lambda x: get_book_details(x['title'], x['author']), axis=1)

# Split the result into columns
unique_books['cover_url'] = details.apply(lambda x: x['cover_url'])
unique_books['published_date'] = details.apply(lambda x: x['published_date'])
unique_books['genre'] = details.apply(lambda x: x['genre'])
unique_books['identifier'] = details.apply(lambda x: x['identifier'])

# Join back to your main highlights dataframe
df_new_cols = df_title_cleaned.merge(unique_books, on=['title', 'author'], how='left')
df_new_cols['date_read'] = pd.Timestamp.now().date()
display(df_new_cols)

# Save Data to Delta Table

In [0]:
spark_df = spark.createDataFrame(df_new_cols)

spark_df.write.format("delta").mode("append").saveAsTable("projects.kindle.highlights")

print(f"Saved {spark_df.count()} highlights to projects.kindle.highlights")

In [0]:
%sql
select * from projects.kindle.highlights